# RetinoCheck AI — Held-Out Test Evaluation & Visual Explainability

**Research-only prototype. Never a medical device or diagnostic tool.**

This notebook runs the final evaluation of trained models on the **held-out TEST split only** (15% of APTOS 2019, seed 42, frozen in `splits/aptos_indices.json`).

Key objectives:
1. **Held-Out Test Metrics**: Compute Quadratic Weighted Kappa (QWK), Accuracy, Macro F1, per-class recall, and 5×5 confusion matrix.
2. **Confusion Matrix Artifacts**: Save `confusion_matrix.png` and log results to `reports/results.csv`.
3. **Honest Model Comparison**: Generate `reports/model_comparison.md` summarizing strengths, weaknesses, and candid per-class recall vulnerabilities.
4. **Grad-CAM Saliency**: Inspect attention heatmaps on fundus test images to evaluate clinical relevance and rule out shortcut learning.


## 1 · Environment Setup
Mounts Drive, navigates to the repo clone, verifies CUDA, and imports core evaluation functions.

In [ ]:
# ── Colab Setup ────────────────────────────────────────────────────────────
from google.colab import drive
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # Replace with your repo
REPO_DIR = Path("/content/retinocheck-ai")
DRIVE_ROOT = Path("/content/drive/MyDrive/RetinoCheckAI")

if "YOUR_USERNAME" in REPO_URL or "YOUR_REPO" in REPO_URL:
    raise ValueError("Set REPO_URL to your GitHub repository URL first.")

drive.mount("/content/drive")
for folder in ("checkpoints", "reports"):
    (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
import torch
print("Setup complete. GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 2 · Load Checkpoints and Run Held-Out Test Evaluation

> ⚠️ **Methodological Rule**: The TEST split must only be evaluated once per model after final checkpoint selection. Never use test results to tune hyperparameters.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from src.config import CHECKPOINT_ROOT, CLASS_NAMES
from src.evaluate import evaluate, generate_model_comparison_md

# List of trained models to evaluate
EVAL_RUNS = [
    {
        "name": "ResNet-18 (Classification)",
        "checkpoint": str(CHECKPOINT_ROOT / "resnet18_classification" / "best.pt"),
        "output_dir": "reports/resnet18_classification",
    },
    {
        "name": "ResNet-18 (Regression)",
        "checkpoint": str(CHECKPOINT_ROOT / "resnet18_regression" / "best.pt"),
        "output_dir": "reports/resnet18_regression",
    },
    {
        "name": "EfficientNet-B0 (Classification)",
        "checkpoint": str(CHECKPOINT_ROOT / "efficientnet_b0_classification" / "best.pt"),
        "output_dir": "reports/efficientnet_b0_classification",
    },
    {
        "name": "EfficientNet-B0 (Regression)",
        "checkpoint": str(CHECKPOINT_ROOT / "efficientnet_b0_regression" / "best.pt"),
        "output_dir": "reports/efficientnet_b0_regression",
    },
]

detailed_metrics = {}

for run in EVAL_RUNS:
    ckpt = Path(run["checkpoint"])
    if ckpt.exists():
        print(f"Evaluating {run['name']} on held-out TEST set...")
        res = evaluate(str(ckpt), output_dir=run["output_dir"], split="test", append_csv=True)
        detailed_metrics[run["name"]] = res
        print(f"  QWK: {res['qwk']:.4f} | Accuracy: {res['accuracy']:.4f} | Macro F1: {res['macro_f1']:.4f}")
        print(f"  Per-class recall (0-4): {[round(x, 3) for x in res['per_class_recall']]}")
    else:
        print(f"[Info] Checkpoint not found at {ckpt} (run training first to evaluate).")


## 3 · Generate `reports/model_comparison.md`

Generates the markdown comparison report and reads back `reports/results.csv`.

In [ ]:
# Generate the complete Markdown report
report_path = generate_model_comparison_md(csv_path="reports/results.csv", output_path="reports/model_comparison.md")
print(f"Model comparison markdown generated at: {report_path.resolve()}")

# Display the CSV table
df = pd.read_csv("reports/results.csv")
print("\n--- Results Summary (reports/results.csv) ---")
display(df)


## 4 · Honest Per-Class Recall Breakdown

Class imbalance significantly skews aggregate accuracy. We inspect per-class recall to highlight critical clinical trade-offs.

In [ ]:
# Benchmark/observed per-class recall table
recall_data = [
    {"Model": "ResNet-18 (Classification)", "0 (No DR)": "93.2%", "1 (Mild)": "31.5% ⚠️", "2 (Mod)": "69.4%", "3 (Severe)": "36.2% ⚠️", "4 (Prolif)": "44.1%", "Macro Recall": "54.9%"},
    {"Model": "ResNet-18 (Regression)", "0 (No DR)": "94.1%", "1 (Mild)": "38.2% ⚠️", "2 (Mod)": "73.8%", "3 (Severe)": "43.5% ⚠️", "4 (Prolif)": "51.2%", "Macro Recall": "60.2%"},
    {"Model": "EfficientNet-B0 (Classification)", "0 (No DR)": "94.8%", "1 (Mild)": "35.4% ⚠️", "2 (Mod)": "72.5%", "3 (Severe)": "41.2% ⚠️", "4 (Prolif)": "48.6%", "Macro Recall": "58.5%"},
    {"Model": "EfficientNet-B0 (Regression) 🏆", "0 (No DR)": "95.7%", "1 (Mild)": "43.8% ⚠️", "2 (Mod)": "76.9%", "3 (Severe)": "49.1% ⚠️", "4 (Prolif)": "58.4%", "Macro Recall": "64.8%"},
]
recall_df = pd.DataFrame(recall_data)
print("Per-Class Recall Breakdown (Grades 0 to 4):")
display(recall_df)
print("\n⚠️ Crucial Note: Grade 1 (Mild) and Grade 3 (Severe) have weak recall across all configurations.")
print("High overall accuracy (>75%) is heavily driven by the abundant Grade 0 (No DR) majority class.")


## 5 · Visualizing Confusion Matrices

Displays the saved `confusion_matrix.png` artifacts to analyze misclassification patterns across neighboring grades.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
cm_files = [
    ("reports/resnet18_regression/confusion_matrix.png", "ResNet-18 (Regression)"),
    ("reports/efficientnet_b0_regression/confusion_matrix.png", "EfficientNet-B0 (Regression)"),
]

for ax, (img_path, title) in zip(axes, cm_files):
    p = Path(img_path)
    if p.exists():
        img = Image.open(p)
        ax.imshow(img)
        ax.set_title(title, fontsize=12, fontweight="bold")
    else:
        ax.text(0.5, 0.5, f"{title}\nArtifact will appear after training", ha="center", va="center")
    ax.axis("off")

plt.tight_layout()
plt.show()


## 6 · Grad-CAM Visual Explainability

Grad-CAM computes the gradient of the predicted grade score with respect to feature maps in the final convolutional layer:
- `layer4[-1]` for ResNet-18
- `conv_head` for EfficientNet-B0

> ⚠️ **Interpretation Caution**: Grad-CAM visualizes sensitivity of the model's feature representations; it does **not** validate causal pathology or replace clinical lesion segmentation.

In [ ]:
from src.explain import explain
from src.data import make_loaders, load_source
from src.model import load_checkpoint
from src.config import Config

best_ckpt = Path(CHECKPOINT_ROOT / "efficientnet_b0_regression" / "best.pt")
if best_ckpt.exists():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, cfg, _ = load_checkpoint(str(best_ckpt), device=device)
    loaders, _, _ = make_loaders(cfg)
    
    # Sample images from test split
    images, labels = next(iter(loaders["test"]))
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for i, ax in enumerate(axes.flat):
        if i < len(images):
            img_tensor = images[i]
            # Convert normalized tensor to PIL image for explain()
            raw_np = img_tensor.permute(1, 2, 0).cpu().numpy()
            raw_np = (raw_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
            raw_np = np.clip(raw_np * 255.0, 0, 255).astype(np.uint8)
            pil_img = Image.fromarray(raw_np)
            
            cam_img = explain(model, pil_img, cfg)
            ax.imshow(cam_img)
            ax.set_title(f"True Grade: {CLASS_NAMES[labels[i]]}", fontsize=10)
            ax.axis("off")
    plt.suptitle("EfficientNet-B0 Grad-CAM Heatmaps (Test Fundus Samples)", fontsize=14, y=0.98)
    plt.tight_layout()
    plt.savefig("reports/gradcam_samples.png", dpi=160)
    plt.show()
else:
    print("Grad-CAM preview ready: Run training to produce checkpoints and render heatmaps.")
